# TruthLens: Exploratory Data Analysis & Preprocessing Pipeline

**Objective**: In this notebook, we analyze the Kaggle Fake and Real News dataset, inspect class balance, evaluate text length and word count distributions, discover vocabulary differences across Fake and Real news articles, and execute an NLP preprocessing and stratified split pipeline.

---

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Add project root to sys.path
sys.path.append('..')
from src.data_loader import ensure_dataset
from src.preprocessor import prepare_and_split_data, TextPreprocessor

sns.set_theme(style="whitegrid", palette="muted")
print("Libraries successfully imported!")

### Step 1: Loading Raw Data
We load both `Fake.csv` and `True.csv` datasets. Each file contains `title`, `text`, `subject`, and `date`.

In [ ]:
df_fake, df_true = ensure_dataset()
print(f"Raw Fake News count: {len(df_fake):,}")
print(f"Raw Real News count: {len(df_true):,}")

### Step 2: Subject Distribution Analysis
Let's observe the distribution of topics/subjects across both classes.

In [ ]:
plt.figure(figsize=(10, 4))
df_fake['subject'].value_counts().plot(kind='bar', color='#e74c3c', title='Fake News Subject Categories')
plt.ylabel('Count')
plt.show()

plt.figure(figsize=(10, 4))
df_true['subject'].value_counts().plot(kind='bar', color='#2ecc71', title='Real News Subject Categories')
plt.ylabel('Count')
plt.show()

### Step 3: Preprocessing & Stratified Splitting (70% Train, 15% Val, 15% Test)
We combine `title` + `text`, eliminate duplicates to prevent data leakage, normalize strings, remove stopwords, and lemmatize tokens.

In [ ]:
df_train, df_val, df_test = prepare_and_split_data(df_fake, df_true)
print(f"Train set: {len(df_train):,} ({len(df_train)/len(df_train+df_val+df_test)*100:.1f}%)")
print(f"Val set:   {len(df_val):,} ({len(df_val)/len(df_train+df_val+df_test)*100:.1f}%)")
print(f"Test set:  {len(df_test):,} ({len(df_test)/len(df_train+df_val+df_test)*100:.1f}%)")

### Step 4: Text Length and Word Count Analysis
Let's inspect the word count distributions between Real and Fake articles.

In [ ]:
df_train['word_count'] = df_train['cleaned_text'].astype(str).apply(lambda x: len(x.split()))

plt.figure(figsize=(10, 5))
sns.histplot(data=df_train, x='word_count', hue='label_name', kde=True, bins=50, palette={'Fake': '#e74c3c', 'Real': '#2ecc71'})
plt.xlim(0, 1500)
plt.title('Word Count Distribution (Real vs Fake News)')
plt.xlabel('Number of Cleaned Words per Article')
plt.show()

### Step 5: Vocabulary & Top N-Grams
Identifying key lexical tokens distinct to each class.

In [ ]:
fake_words = ' '.join(df_train[df_train['label'] == 1]['cleaned_text']).split()
real_words = ' '.join(df_train[df_train['label'] == 0]['cleaned_text']).split()

top_fake = Counter(fake_words).most_common(15)
top_real = Counter(real_words).most_common(15)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
pd.DataFrame(top_fake, columns=['Word', 'Count']).plot(kind='barh', x='Word', y='Count', color='#e74c3c', ax=axes[0], title='Top 15 Words (Fake News)')
pd.DataFrame(top_real, columns=['Word', 'Count']).plot(kind='barh', x='Word', y='Count', color='#2ecc71', ax=axes[1], title='Top 15 Words (Real News)')
plt.tight_layout()
plt.show()

### Phase 1 Summary & Conclusion
- **Data Integrity**: Handled missing values, deduplicated identical articles to prevent data leakage, and combined `title` + `text` for complete context.
- **Class Balance**: The dataset is balanced (~52% Fake, ~48% Real), ensuring models won't be biased towards a dominant majority class.
- **Splits Saved**: `data/processed/train.csv`, `val.csv`, `test.csv` ready for Phase 2 baseline model training.